In [1]:
!pip install cachetools

### TLRU Test

In [6]:
from typing import Any
from cachetools import TLRUCache
import time


def ttu(_key: Any, _value: Any, now: float):
    return now + 5  # 5 seconds from now


cache = TLRUCache(maxsize=5, ttu=ttu)

cache["a"] = 1
cache["b"] = 2

print("Initial cache keys:", list(cache.keys()))

print("Waiting for 3 seconds...")
time.sleep(3)
print("Value of 'a' after refresh:", cache["a"])
print("After accessing 'a', cache keys:", list(cache.keys()))

print("Waiting for 3 seconds...")
time.sleep(3)
print("After expiration, cache keys:", list(cache.keys()))

Initial cache keys: ['a', 'b']
Waiting for 3 seconds...
Value of 'a' after refresh: 1
After accessing 'a', cache keys: ['a', 'b']
Waiting for 3 seconds...
After expiration, cache keys: []


### Expire After Access



In [ ]:
import time
from collections.abc import Callable
from typing import Any

from cachetools import TTLCache

# TTU: (key, value, now) -> expiration_time
TTUCallable = Callable[[Any, Any, float], float]


class TTICache(TTLCache[Any, Any]):
    def __init__(
        self,
        maxsize: int,
        ttl: float,
        ttu: TTUCallable,
        timer: Callable[[], float] = time.monotonic,
        getsizeof: Callable[[Any], int] | None = None,
    ) -> None:
        super().__init__(maxsize, ttl, timer, getsizeof)
        self.__ttu = ttu

    def __getitem__(
        self,
        key: Any,
        cache_getitem: Callable[..., Any] = TTLCache.__getitem__,  # type: ignore[assignment]
    ) -> Any:
        # 1. Call the superclass to get the value (if expired, superclass will raise KeyError or handle deletion)
        value = cache_getitem(self, key)

        # 2. If the value is retrieved, it has not absolutely expired; now execute TTU (Time To Use) logic
        now = self.timer()
        new_expiration = self.__ttu(key, value, now)

        # 3. Update the expiration time for this key
        # cachetools internally uses self._TTLCache__links to store expiration info
        # We need to update the link object's expires property and move it to the end of the queue (expire last)
        link = self._TTLCache__links[key]  # pyright: ignore[reportAttributeAccessIssue]
        link.expires = new_expiration

        # Move the link from its current position to the tail of the doubly-linked list
        # This ensures that the expiration order matches the linked list order
        self._update_link(link)

        return value

    def _update_link(self, link: TTLCache._Link) -> None:  # pyright: ignore[reportAttributeAccessIssue]
        """Move the link to the end of the doubly linked list, maintaining expiration order"""
        root = self._TTLCache__root  # pyright: ignore[reportAttributeAccessIssue]
        links = self._TTLCache__links  # pyright: ignore[reportAttributeAccessIssue]
        # 1. Unlink from current position
        link.prev.next = link.next
        link.next.prev = link.prev
        # 2. Insert at tail (before root)
        link.next = root
        link.prev = root.prev
        link.prev.next = link
        root.prev = link
        # 3. Keep OrderedDict in sync with linked list order
        links.move_to_end(link.key)

### Example

In [ ]:
def ttu(_key: Any, _value: Any, now: float) -> float:
    return now + 5  # 5 seconds from now


cache = TTICache(maxsize=5, ttl=5, ttu=ttu)

cache["a"] = 1
cache["b"] = 2

print("Initial cache keys:", list(cache.keys()))

print("Waiting for 3 seconds...")
time.sleep(3)
print("Value of 'a' after refresh:", cache["a"])
print("After accessing 'a', cache keys:", list(cache.keys()))

print("Waiting for 3 seconds...")
time.sleep(3)
print("After expiration, cache keys:", list(cache.keys()))

Initial cache keys: ['a', 'b']
Waiting for 3 seconds...
Value of 'a' after refresh: 1
After accessing 'a', cache keys: ['b', 'a']
Waiting for 3 seconds...
After expiration, cache keys: ['a']
